# TP3 - Algoritmos Genéticos

## Objetivo y ejecución

Resolver el acertijo de cinco casas mediante un algoritmo genético y responder quién usa Vim en la solución encontrada. Cada casa tiene un color, una profesión, un lenguaje, una base de datos y un editor distintos. Se deben cumplir las quince pistas de la consigna.

En **Jupyter**, abra este notebook en la misma carpeta que `ga.py` y `verificar.py`, con un kernel de Python 3.10 o posterior. Reinicie el kernel y ejecute todas las celdas en orden.

En **Google Colab**, suba el notebook y luego cargue `ga.py` y `verificar.py` mediante el panel **Archivos**. No se necesitan paquetes adicionales. El notebook reutiliza los scripts; la enumeración exacta solo se consulta al final como comprobación independiente.

## Representación

Cada individuo contiene cinco permutaciones de las posiciones 0 a 4. El orden de los atributos está definido en `CATEGORIES`; cada número indica la casa de ese atributo. Se fijan Desarrollador en la primera casa, Notepad++ en la central y Azul en la primera, por las pistas 10, 9 y la combinación de 10 con 15. Las demás posiciones se sortean.

In [1]:
from pathlib import Path
from random import Random
from dataclasses import replace

necesarios = ("ga.py", "verificar.py")
faltantes = [nombre for nombre in necesarios if not Path(nombre).is_file()]
assert not faltantes, "Cargue estos archivos en la carpeta de trabajo: " + ", ".join(faltantes)

from ga import (
    CATEGORIES, HEADERS, Config, random_individual, well_formed,
    fitness, tournament, crossover, mutate, solve, format_result,
    clue_status, is_solution, vim_user,
)

config = Config(seed=42)
candidato = random_individual(Random(config.seed))
assert well_formed(candidato)
for categoria, atributos, posiciones in zip(HEADERS, CATEGORIES, candidato):
    asignaciones = ", ".join(f"{atributo}: casa {pos + 1}" for atributo, pos in zip(atributos, posiciones))
    print(f"{categoria}: {asignaciones}")

Color: Rojo: casa 4, Azul: casa 1, Verde: casa 3, Blanco: casa 5, Amarillo: casa 2
Profesión: Matemático: casa 5, Hacker: casa 4, Ingeniero: casa 2, Analista: casa 3, Desarrollador: casa 1
Lenguaje: Python: casa 4, C#: casa 2, Java: casa 3, C++: casa 5, JavaScript: casa 1
Base de datos: Cassandra: casa 2, MongoDB: casa 3, Neo4j: casa 4, Redis: casa 5, HBase: casa 1
Editor: Brackets: casa 5, Sublime Text: casa 1, Atom: casa 4, Notepad++: casa 3, Vim: casa 2


## Población y aptitud

La población inicial se genera mediante barajados aleatorios. El fitness cuenta las pistas satisfechas: equivale a 15 menos una penalización de un punto por cada incumplimiento. Un candidato estructuralmente válido todavía puede incumplir pistas. La configuración real utiliza 300 individuos; este ejemplo muestra ocho para facilitar la lectura.

In [2]:
rng = Random(17)
poblacion = [random_individual(rng) for _ in range(8)]
aptitudes = [fitness(individuo, config.immediate) for individuo in poblacion]
assert all(well_formed(individuo) for individuo in poblacion)
print("Población de la búsqueda:", config.population)
print("Pistas cumplidas por los ocho candidatos de ejemplo:", aptitudes)
print("Fitness máximo posible: 15")

Población de la búsqueda: 300
Pistas cumplidas por los ocho candidatos de ejemplo: [8, 5, 8, 6, 7, 7, 5, 7]
Fitness máximo posible: 15


## Operadores genéticos

La selección usa torneos de tres candidatos. El cruzamiento OX conserva un segmento de un padre y completa la permutación según el orden del otro. La mutación intercambia dos genes libres, sin perder ni duplicar atributos.

En el ejemplo se fuerza una mutación por categoría (`rate=1`) para observar el intercambio; la búsqueda real mantiene su tasa de **0,12** por categoría y probabilidad de cruce **0,30**. Una operación aislada no garantiza mejorar el fitness.

La evolución conserva dos élites, incorpora quince inmigrantes por generación y reinicializa la población tras cien generaciones sin mejora global. El mejor candidato global se reserva para informar el resultado.

In [3]:
padre_a = tournament(poblacion, aptitudes, rng, config.tournament_size)
padre_b = tournament(poblacion, aptitudes, rng, config.tournament_size)
hijo = crossover(padre_a, padre_b, rng)
mutado = mutate(hijo, rng, rate=1)
assert all(well_formed(individuo) for individuo in (padre_a, padre_b, hijo, mutado))
assert padre_a in poblacion and padre_b in poblacion
print("Fitness de padre A, padre B, hijo y mutado:",
      [fitness(individuo) for individuo in (padre_a, padre_b, hijo, mutado)])
print("Posiciones de colores antes del swap:", hijo[0])
print("Posiciones de colores después del swap:", mutado[0])
print("Todos conservan cinco valores únicos por categoría.")

Fitness de padre A, padre B, hijo y mutado: [8, 8, 8, 7]
Posiciones de colores antes del swap: (4, 0, 3, 2, 1)
Posiciones de colores después del swap: (4, 0, 1, 2, 3)
Todos conservan cinco valores únicos por categoría.


## Resolución mediante el algoritmo genético

Se ejecuta `solve` con semilla 42 y la configuración predeterminada. Hay tres condiciones de corte: solución completa, máximo de 3000 generaciones o máximo de 30 segundos medido con un reloj monotónico. Los límites son configurables.

Por defecto, verde debe estar en una posición mayor que blanca, sin exigir adyacencia. La salida distingue una solución completa del mejor resultado parcial y solo responde sobre Vim si se cumplen todas las pistas.

In [4]:
resultado = solve(config)
print(format_result(resultado, config))

Semilla: 42 | Derecha: cualquier posición mayor
Corte: solucion | Generación: 9 | Tiempo: 0.085 s
Pistas cumplidas: 15/15
SOLUCIÓN COMPLETA
Casa | Color    | Profesión     | Lenguaje   | Base de datos | Editor      
1    | Azul     | Desarrollador | C++        | Neo4j         | Sublime Text
2    | Rojo     | Matemático    | Java       | Redis         | Vim         
3    | Blanco   | Hacker        | Python     | HBase         | Notepad++   
4    | Amarillo | Analista      | JavaScript | Cassandra     | Atom        
5    | Verde    | Ingeniero     | C#         | MongoDB       | Brackets    
En ESTA solución utiliza Vim: Matemático.
El enunciado admite otras soluciones y distintos usuarios de Vim; ver verificar.py.


## Validación de las quince pistas

Se comprueba la estructura y se vuelven a evaluar todas las condiciones antes de aceptar el resultado. El fitness y la validación comparten los predicados de `clue_status`; las pruebas y la enumeración separada aportan comprobaciones adicionales.

In [5]:
pistas = (
    "Cinco casas y valores únicos por categoría",
    "Matemático en la casa roja", "Hacker programa en Python",
    "Brackets en la casa verde", "Analista usa Atom",
    "Verde a la derecha de blanca", "Redis y Java en la misma casa",
    "Cassandra en la casa amarilla", "Notepad++ en la casa central",
    "Desarrollador en la primera casa", "HBase vecino de JavaScript",
    "Cassandra vecino de C#", "Neo4j y Sublime Text en la misma casa",
    "Ingeniero usa MongoDB", "Desarrollador en la casa azul",
)
estados = clue_status(resultado.individual, config.immediate)
for numero, (pista, cumple) in enumerate(zip(pistas, estados), 1):
    print(f"{numero:2}. {'Cumple' if cumple else 'No cumple'}: {pista}")
assert is_solution(resultado.individual, config.immediate), "El presupuesto solo permitió un resultado parcial."
assert all(estados)
print("Validación completa:", sum(estados), "/ 15")
print("En esta solución usa Vim:", vim_user(resultado.individual, config.immediate))

 1. Cumple: Cinco casas y valores únicos por categoría
 2. Cumple: Matemático en la casa roja
 3. Cumple: Hacker programa en Python
 4. Cumple: Brackets en la casa verde
 5. Cumple: Analista usa Atom
 6. Cumple: Verde a la derecha de blanca
 7. Cumple: Redis y Java en la misma casa
 8. Cumple: Cassandra en la casa amarilla
 9. Cumple: Notepad++ en la casa central
10. Cumple: Desarrollador en la primera casa
11. Cumple: HBase vecino de JavaScript
12. Cumple: Cassandra vecino de C#
13. Cumple: Neo4j y Sublime Text en la misma casa
14. Cumple: Ingeniero usa MongoDB
15. Cumple: Desarrollador en la casa azul
Validación completa: 15 / 15
En esta solución usa Vim: Matemático


## Ambigüedad y limitaciones

La consigna dice «a la derecha» sin especificar «inmediatamente». La opción inmediata exige adyacencia. Se ejecuta también ese modo y, **después** de las búsquedas genéticas, se usa `verificar.py` para enumerar las soluciones como comprobación independiente. Nunca se utiliza esa enumeración para alimentar el AG.

Las pistas admiten varios usuarios de Vim. La profesión mostrada corresponde a una solución concreta. El éxito con estas semillas no garantiza convergencia para otros presupuestos o semillas.

In [6]:
config_inmediata = replace(config, immediate=True)
resultado_inmediato = solve(config_inmediata)
assert is_solution(resultado_inmediato.individual, True)
print("Modo inmediato:", resultado_inmediato.score, "/ 15; generación", resultado_inmediato.generation)
print("En esa solución usa Vim:", vim_user(resultado_inmediato.individual, True))

from verificar import summary

for inmediata, solucion in ((False, resultado), (True, resultado_inmediato)):
    soluciones_exactas, usuarios = summary(inmediata)
    assert solucion.individual in soluciones_exactas
    modo = "inmediata" if inmediata else "cualquier posición mayor"
    print(f"Derecha {modo}: {len(soluciones_exactas)} soluciones completas.")
    print("  Profesiones que pueden usar Vim:", ", ".join(sorted(usuarios)))

Modo inmediato: 15 / 15; generación 22
En esa solución usa Vim: Desarrollador
Derecha cualquier posición mayor: 31 soluciones completas.
  Profesiones que pueden usar Vim: Desarrollador, Hacker, Ingeniero, Matemático
Derecha inmediata: 14 soluciones completas.
  Profesiones que pueden usar Vim: Desarrollador, Hacker, Ingeniero, Matemático


## Conclusión

La demostración mantiene una única implementación del algoritmo, conserva la estructura de los individuos y valida las quince pistas de las soluciones obtenidas. La enumeración independiente confirma que el acertijo no determina una respuesta universal sobre Vim.

Como referencias conceptuales se consultaron los ejemplos de clase de Einstein y de la mochila y la presentación de algoritmos genéticos. Las pistas utilizadas aquí son las del TP3; no se reemplazan por las del acertijo clásico.